# TP 2 — Comparer ML et deep learning sur Tiny Shakespeare

Je compare machine learning et deep learning sur l'identification du locuteur dans Tiny Shakespeare. Je pars du TP1 et de ses analyses, et je garde son protocole d'évaluation, car les réseaux changent la représentation et le modèle, mais pas la façon de mesurer.

| Étape | Section | Ce que je livre |
|---|---|---|
| 1. Nettoyage et prétraitement | section 1 | Les 962 répliques du TP1, le découpage, puis un tokenizer commun aux modèles ML et DL. |
| 2. Représentation | section 2 | La référence TF–IDF du TP1 et des embeddings appris avec le classifieur. |
| 3. Comparaison ML / DL | section 3 | Classe majoritaire, LinearSVC, puis MLP, LSTM et GRU sur les mêmes cinq folds. |
| 4. Évaluation | section 4 | Scores par fold, erreurs relues réplique par réplique, expériences ciblées, test final, journal et conclusion. |


## 0. Mon problème et les acquis du TP1

Je prédis le locuteur d'une réplique à partir de son seul texte, parmi 10 classes. Comme au TP1, je garde les répliques d'au moins 10 mots des 10 locuteurs les plus fréquents.

| Fait observé dans le TP1 | Ce que j'examine dans le TP2 |
|---|---|
| 962 répliques, de 78 à 129 par classe | Le sur-apprentissage, car le jeu est petit et modérément déséquilibré. |
| 9 confusions JULIET–ROMEO (4 + 5) et 7 de GLOUCESTER vers DUKE VINCENTIO sur le test | Les indices communs aux erreurs, dont la pièce d'origine. |
| `kate` (2,645), `romeo` (1,623), `paulina` (1,447) et `camillo` (1,211) portent les poids les plus élevés (TP1, section 8.3) | Le modèle reconnaît-il des personnages, donc une pièce, ou un style ? |
| LinearSVC obtient un F1 macro de 0,482 ± 0,035 en validation croisée, 1 sur le train des folds et 0,563 sur le test | La généralisation plutôt que l'ajustement au train. Le 0,563 sert de repère dans la section 4.4. |


> *Mes hypothèses :*
>
> - **H1, indices lexicaux.** Un MLP sur la moyenne des embeddings reste à moins de 0,05 de F1 macro de TF–IDF + LinearSVC, car les deux ignorent l'ordre des mots. Testée dans la section 3.4.
> - **H2, ordre des mots.** LSTM et GRU ne dépassent le MLP que si l'ordre porte un signal, donc mélanger les mots doit les dégrader mais pas le MLP. Testée dans les sections 3.5, 3.6 et 4.3.
> - **H3, mémorisation.** Sans régularisation, un réseau atteint un F1 macro train proche de 1 en moins de 20 epochs, avec un écart train/validation supérieur à 0,3. Testée par les courbes des sections 3.4 à 3.6.
> - **H4, coût.** À largeur égale, le GRU a moins de paramètres et s'entraîne plus vite que le LSTM pour un F1 macro comparable. Testée dans les sections 3.6 et 4.1.

Je décide sur les scores moyens, leur dispersion et le coût, avant de regarder le test.


### Mon environnement

Je sélectionne `.venv/bin/python` comme kernel et j'installe les dépendances dans le terminal :

```sh
uv pip install --python .venv/bin/python -r requirements.txt
```

> *Mes choix :*
>
> - PyTorch, pour écrire moi-même forward, loss, backward et mise à jour des poids.
> - GPU Apple par MPS, le backend GPU de PyTorch sur Mac. Un M3 Max suffit pour des réseaux de quelques centaines de milliers de paramètres sur 962 répliques, et les versions sont figées dans `requirements.txt`.
> - Seed 42, la graine du générateur aléatoire, réinitialisée avant chaque essai par une fonction appelée à chaque fold (section 3.1). Je mesure la variabilité par la dispersion entre folds et je vérifie deux runs à seed égale.


In [1]:
# Environnement : versions, matériel, device PyTorch et seed unique pour random, numpy et torch.
import platform
import random
import subprocess
import sys
from importlib.metadata import version

import numpy as np
import torch

# Seed fixée ici ; la fonction de la section 3.1 la réinitialisera avant chaque entraînement.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# GPU Apple par MPS si disponible, sinon CPU
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Versions et dépendances
print(f'Python : {platform.python_version()} — {sys.executable}')
for paquet in ('numpy', 'pandas', 'matplotlib', 'scikit-learn', 'ipykernel', 'torch'):
    print(f'{paquet} : {version(paquet)}')

# Système et processeur
print(f'Système : {platform.platform()} — {platform.machine()}')
if sys.platform == 'darwin':
    processeur, ram = subprocess.check_output(
        ['sysctl', '-n', 'machdep.cpu.brand_string', 'hw.memsize'], text=True,
    ).splitlines()
    print(f'Processeur : {processeur} — RAM : {int(ram) / 2**30:g} Gio')

print(f'Device : {device} — Seed : {RANDOM_STATE}')


Python : 3.14.6 — /Users/hopy/Documents/ING3_GIA/ING3_IA_GB_S1/NLP/.venv/bin/python
numpy : 2.5.3
pandas : 3.0.5
matplotlib : 3.11.1
scikit-learn : 1.9.0
ipykernel : 7.3.0
torch : 2.14.0
Système : macOS-26.5-arm64-arm-64bit-Mach-O — arm64
Processeur : Apple M3 Max — RAM : 36 Gio
Device : mps — Seed : 42


## 1. Nettoyage et prétraitement

### 1.1 Retrouver les mêmes répliques

Je reprends le chargement et le parser du TP1 sans dépendre de son état d'exécution. Le nom en tête de bloc donne l'étiquette et ne reste pas dans l'entrée du modèle. Le parser est mon seul nettoyage, car je garde ponctuation et majuscules pour laisser le tokenizer de la section 1.3 décider, mesure à l'appui.


In [2]:
# Corpus : téléchargement de Tiny Shakespeare et parser du TP1 (bloc = étiquette + réplique).
import re
import urllib.request

import pandas as pd
from IPython.display import display

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Téléchargement du texte brut
with urllib.request.urlopen(URL, timeout=30) as reponse:
    texte = reponse.read().decode('utf-8')


def parser_repliques(txt: str) -> pd.DataFrame:
    """Reprend les règles du TP1 : un bloc, une étiquette et sa réplique."""
    lignes = []
    for bloc in txt.split("\n\n"):
        bloc = bloc.strip()
        if not bloc:
            continue
        parties = bloc.split("\n")
        entete = re.match(r"^([A-Z][A-Za-z' ]*):$", parties[0])  # Nom en majuscules puis « : »
        if entete and len(parties) > 1:
            lignes.append({
                'locuteur': entete.group(1).strip(),
                'replique': " ".join(p.strip() for p in parties[1:]).strip(),
            })
    return pd.DataFrame(lignes)


df = parser_repliques(texte)


In [3]:
# Jeu retenu : répliques d'au moins 10 mots, puis les 10 locuteurs les plus fréquents, comme au TP1.
seuil, nb_locuteurs = 10, 10

# Le seuil de longueur s'applique avant de compter les locuteurs.
longueurs = df['replique'].str.split().str.len()
locuteurs = df.loc[longueurs >= seuil, 'locuteur'].value_counts().head(nb_locuteurs).index

donnees = df.loc[
    (longueurs >= seuil) & df['locuteur'].isin(locuteurs),
    ['replique', 'locuteur'],
].copy()  # Ordre et indices du TP1 conservés.


In [4]:
# Contrôles du jeu retenu : effectifs, longueurs, valeurs manquantes, doublons.
effectifs = donnees['locuteur'].value_counts()
longueurs_retenues = donnees['replique'].str.split().str.len()

# Comptes globaux
print(f'Extraction : {len(df):,} répliques, {df["locuteur"].nunique()} labels')
print(f'Jeu retenu : {len(donnees)} répliques, {len(effectifs)} classes')
print(f'Longueurs (mots) : min {longueurs_retenues.min()}, '
      f'médiane {longueurs_retenues.median():g}, max {longueurs_retenues.max()}')
print(f'Valeurs manquantes : {donnees.isna().sum().sum()}')
print(f'Répliques vides : {donnees["replique"].str.strip().eq("").sum()}')
print(f'Textes dupliqués dans le jeu retenu : {donnees["replique"].duplicated().sum()}')

# Effectifs par classe et aperçu
display(effectifs.rename('Répliques').to_frame())
display(donnees.head(3))


Extraction : 7,094 répliques, 297 labels
Jeu retenu : 962 répliques, 10 classes
Longueurs (mots) : min 10, médiane 31, max 578
Valeurs manquantes : 0
Répliques vides : 0
Textes dupliqués dans le jeu retenu : 0


,Répliques
locuteur,
GLOUCESTER,129
DUKE VINCENTIO,125
MENENIUS,107
PETRUCHIO,95
ROMEO,94
CORIOLANUS,87
JULIET,85
LEONTES,82
ISABELLA,80


,replique,locuteur
22,"What work's, my countrymen, in hand? where go ...",MENENIUS
24,"Why, masters, my good friends, mine honest nei...",MENENIUS
26,"I tell you, friends, most charitable care Have...",MENENIUS


> *Mes observations :*
>
> Je retrouve les 962 répliques du TP1, car l'extraction donne 7 094 répliques et 297 étiquettes, puis 10 classes de 78 à 129 répliques après le seuil de 10 mots. Aucune valeur manquante, aucune réplique vide, aucun doublon.
>
> Les longueurs vont de 10 à 578 mots, avec une médiane de 31. Cette asymétrie fixe un réglage de la section 2.2, car une longueur maximale égale au maximum remplirait les batches de padding pour quelques répliques.


### 1.2 Séparer apprentissage, sélection et évaluation

Je reproduis le découpage stratifié du TP1, soit 769 répliques de développement et 193 de test avec la seed 42. Mêmes données, même seed et même version de scikit-learn donnent le même découpage, et la section 1.3 le vérifie avec la taille du vocabulaire TF–IDF du TP1.


In [5]:
# Découpage développement / test du TP1 : 80/20 stratifié, seed 42.
from sklearn.model_selection import StratifiedKFold, train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    donnees['replique'], donnees['locuteur'],
    test_size=0.2, stratify=donnees['locuteur'], random_state=RANDOM_STATE,
)

# Vérifications : aucun indice ni texte partagé entre développement et test.
assert set(X_train.index).isdisjoint(X_test.index)
assert not set(X_train).intersection(X_test)

# Tailles, empreinte du découpage et répartition des classes
print(f'Développement : {len(X_train)} répliques ; test : {len(X_test)}')
print(f'Premiers indices du test : {list(X_test.index[:10])}')
display(pd.DataFrame({
    'Développement': y_train.value_counts(),
    'Test': y_test.value_counts(),
}).sort_values('Développement', ascending=False))


Développement : 769 répliques ; test : 193
Premiers indices du test : [3009, 6783, 5903, 1321, 3338, 2294, 3344, 2875, 3378, 2212]


,Développement,Test
locuteur,,
GLOUCESTER,103,26
DUKE VINCENTIO,100,25
MENENIUS,85,22
PETRUCHIO,76,19
ROMEO,75,19
CORIOLANUS,70,17
JULIET,68,17
LEONTES,66,16
ISABELLA,64,16


Sur les 769 répliques, je prépare cinq folds stratifiés avec `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`. Tous les modèles utilisent les mêmes folds, donc je les compare sur les mêmes répliques.

| Ensemble | Taille | Rôle |
|---|---|---|
| Train du fold | 615 ou 616 | Tout ce qui dépend des données s'y apprend, donc vocabulaire, IDF, embeddings et poids. |
| Validation du fold | 154 ou 153 | Comparaison des modèles et choix des réglages, dont le nombre d'epochs des réseaux (section 3.1). |
| Test | 193 | Une seule évaluation, après avoir figé tous les choix (section 4.4). |

Je ne réserve pas de validation interne aux réseaux, car la validation du fold suffit pour choisir leur nombre d'epochs, comme expliqué dans la section 3.1.


In [6]:
# Cinq folds stratifiés sur les 769 répliques de développement, communs aux modèles ML et DL.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Indices d'origine, utilisables avec .loc, plutôt que positions
folds = [
    {'train': X_train.index[positions_train], 'validation': X_train.index[positions_validation]}
    for positions_train, positions_validation in cv.split(X_train, y_train)
]

# Vérification : les dix classes sont présentes dans chaque validation.
assert all(y_train.loc[fold['validation']].nunique() == y_train.nunique() for fold in folds)

display(pd.DataFrame(
    [{nom: len(indices) for nom, indices in fold.items()} for fold in folds],
    index=pd.RangeIndex(1, len(folds) + 1, name='Fold'),
))


,train,validation
Fold,,
1,615,154
2,615,154
3,615,154
4,615,154
5,616,153


> *Mon analyse :*
>
> Je retrouve 769 répliques de développement et 193 de test, comme au TP1, et les deux vérifications `assert` passent, donc aucun indice ni texte n'est partagé. La section 1.3 ajoutera une preuve, car le TF–IDF du TP1 recalculé sur ces 769 répliques doit retrouver ses 5 915 termes.
>
> Les cinq folds donnent 615 répliques d'apprentissage et 154 de validation pour les folds 1 à 4, puis 616 et 153 pour le dernier. Chaque réplique de développement est évaluée une fois, et les dix classes sont présentes dans chaque validation.


### 1.3 Choisir le tokenizer

Je tokenise en mots, car avec 962 répliques un vocabulaire de quelques milliers de tokens distincts reste apprenable et les séquences restent courtes, avec une médiane de 31 mots. Avant de fixer les règles, je compte les cas rencontrés dans les 769 répliques de développement, pour décider sur des chiffres.


In [7]:
# Faits sur le texte brut des 769 répliques de développement, avant toute décision de découpage.
from collections import Counter

# Chaque cas est un motif regex ; « mot » = lettres et apostrophes, sans découpage imposé.
CAS = {
    'Apostrophe en tête': r"(?<![A-Za-z])'[A-Za-z]+",
    'Apostrophe interne': r"[A-Za-z]+'[A-Za-z]+",
    'Apostrophe en fin': r"[A-Za-z]+'(?![A-Za-z])",
    'Mot d’une lettre': r"(?<![A-Za-z'])[A-Za-z](?![A-Za-z'])",
    'Tiret': r"[A-Za-z]+-[A-Za-z]+",
    'Chiffre': r"\d+",
    'Ponctuation': r"[,;:!?.]",
}


def compter_cas(motif: str) -> dict:
    """Répliques contenant le motif et les trois formes les plus fréquentes qu'il capture."""
    expression = re.compile(motif)
    formes = Counter(forme.lower() for texte in X_train for forme in expression.findall(texte))
    return {
        'Répliques concernées': sum(bool(expression.search(texte)) for texte in X_train),
        'Formes les plus fréquentes': ', '.join(
            f'{forme} ({n})' for forme, n in formes.most_common(3)),
    }


# Un tableau, une ligne par cas
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame({cas: compter_cas(motif) for cas, motif in CAS.items()}).T
            .rename_axis('Cas dans le texte brut'))

# Longueurs médianes en mots et en caractères ; leur rapport chiffre un découpage en caractères.
mots_par_replique = X_train.str.split().str.len()
caracteres_par_replique = X_train.str.len()
caracteres_par_mot = caracteres_par_replique.median() / mots_par_replique.median()
print(f'Longueur médiane : {mots_par_replique.median():g} mots '
      f'et {caracteres_par_replique.median():g} caractères par réplique, '
      f'soit {caracteres_par_mot:.1f} caractères par mot.')


,Répliques concernées,Formes les plus fréquentes
Cas dans le texte brut,,
Apostrophe en tête,119,"'tis (61), 'twere (10), 'twas (8)"
Apostrophe interne,392,"i'll (103), that's (20), ne'er (13)"
Apostrophe en fin,81,"i' (23), o' (22), banished' (4)"
Mot d’une lettre,549,"i (902), a (574), o (128)"
Tiret,149,"to-morrow (15), to-night (7), to-day (7)"
Chiffre,3,3 (3)
Ponctuation,769,", (3783), . (1247), ; (730)"


Longueur médiane : 31 mots et 159 caractères par réplique, soit 5.1 caractères par mot.


> *Mes observations :*
>
> L'apostrophe interne touche 392 répliques, avec « i'll » (103), « that's » (20) et « ne'er » (13) en tête. L'apostrophe en tête touche 119 répliques, surtout « 'tis » (61), et l'apostrophe en fin 81 répliques, surtout « i' » (23) et « o' » (22). Les mots d'une lettre apparaissent dans 549 répliques, avec « i » (902), « a » (574) et « o » (128). Le tiret relie deux mots dans 149 répliques, les chiffres touchent 3 répliques et la ponctuation est partout. Une réplique médiane compte 31 mots pour 159 caractères, soit 5,1 caractères par mot, donc un découpage en caractères donnerait des séquences cinq fois plus longues.


Le TP1 utilisait le `token_pattern` par défaut de scikit-learn, `(?u)\b\w\w+\b`, qui supprime les tokens d'une lettre et coupe aux apostrophes. « I'll » devenait `ll` et « I » disparaissait, comme le montraient les IDF les plus élevés du TP1 avec `ll bring`. Or « I », « not » ou « Romeo » peuvent être des indices de locuteur. Je définis donc un tokenizer unique pour TF–IDF et pour les réseaux, avec ces règles :

| Cas dans le texte | Décision | Pourquoi |
|---|---|---|
| Majuscules (« Kate », « Romeo ») | minuscules | Même indice en début et en milieu de phrase, comme au TP1. |
| Ponctuation | supprimée | Un séparateur présent dans les 769 répliques, et aucune de mes hypothèses ne porte sur elle. |
| Mots d'une lettre (« I », « a », « O ») | conservés | Pronom, article et interjection, dans 549 répliques. |
| Apostrophe interne (« I'll », « that's ») | un seul token | 392 répliques, car couper « I'll » (103) donne le fragment `ll`. |
| Apostrophe en tête ou en fin (« 'tis », « i' ») | retirée | « 'tis » (61) devient `tis` et « i' » (23) devient `i`, car je garde le mot. |
| Tiret (« to-morrow ») | sépare | 149 répliques, et `to` puis `morrow` restent lisibles. |
| Chiffres | supprimés | 3 répliques seulement. |
| Caractères comme unités | non | 5,1 caractères par mot, donc des séquences cinq fois plus longues pour un petit réseau récurrent. |

Je garde aussi la référence exacte du TP1 dans la section 3.3, car l'écart entre les deux TF–IDF mesurera l'effet du tokenizer seul.


In [8]:
# Tokenizer commun aux modèles ML et DL, et reproduction du découpage par défaut du TP1.
MOTIF_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")  # Lettres et apostrophes internes.
MOTIF_DEFAUT = re.compile(r"(?u)\b\w\w+\b")  # token_pattern par défaut de scikit-learn (TP1).


def tokeniser(texte: str) -> list[str]:
    """Tokenizer commun : minuscules, mots et contractions."""
    return MOTIF_TOKEN.findall(texte.lower())


def tokeniser_defaut(texte: str) -> list[str]:
    """Découpage du TP1, pour mesurer ce que je change."""
    return MOTIF_DEFAUT.findall(texte.lower())


def montrer_decoupages(indices: tuple[int, ...]) -> None:
    """Affiche des répliques de développement et leurs deux découpages."""
    for indice in indices:
        print(f'{y_train.loc[indice]} : {X_train.loc[indice]}')
        print(f'   TP1    : {tokeniser_defaut(X_train.loc[indice])}')
        print(f'   commun : {tokeniser(X_train.loc[indice])}\n')


# Trois répliques : contraction, apostrophe en tête, nom propre et mots d'une lettre.
montrer_decoupages((3062, 563, 6253))


ROMEO : Switch and spurs, switch and spurs; or I'll cry a match.
   TP1    : ['switch', 'and', 'spurs', 'switch', 'and', 'spurs', 'or', 'll', 'cry', 'match']
   commun : ['switch', 'and', 'spurs', 'switch', 'and', 'spurs', 'or', "i'll", 'cry', 'a', 'match']

MENENIUS : For 'tis a sore upon us, You cannot tent yourself: be gone, beseech you.
   TP1    : ['for', 'tis', 'sore', 'upon', 'us', 'you', 'cannot', 'tent', 'yourself', 'be', 'gone', 'beseech', 'you']
   commun : ['for', 'tis', 'a', 'sore', 'upon', 'us', 'you', 'cannot', 'tent', 'yourself', 'be', 'gone', 'beseech', 'you']

PETRUCHIO : Nay, hear you, Kate: in sooth you scape not so.
   TP1    : ['nay', 'hear', 'you', 'kate', 'in', 'sooth', 'you', 'scape', 'not', 'so']
   commun : ['nay', 'hear', 'you', 'kate', 'in', 'sooth', 'you', 'scape', 'not', 'so']



Je compare maintenant les deux découpages sur les 769 répliques de développement, pour mesurer ce que le tokenizer commun ajoute et ce qu'il retire. Je recalcule aussi le TF–IDF du TP1 sur ces répliques, car retrouver ses 5 915 termes prouve que ce sont les mêmes 769 répliques.


In [9]:
# Ce que chaque découpage voit sur les 769 répliques de développement, puis l'empreinte du TP1.
from sklearn.feature_extraction.text import TfidfVectorizer

TOKENISEURS = {'TP1': tokeniser_defaut, 'commun': tokeniser}
PARAMS_TFIDF = dict(ngram_range=(1, 2), min_df=2, max_df=1.0, stop_words=None, sublinear_tf=True)


def compter_tokens(decouper) -> Counter:
    """Fréquence de chaque token sur le développement : une description, pas un apprentissage."""
    return Counter(token for texte in X_train for token in decouper(texte))


def plus_frequents(comptes: dict, n: int = 6) -> str:
    """Les n tokens les plus fréquents d'un dictionnaire de comptes, avec leur compte."""
    tries = sorted(comptes.items(), key=lambda paire: -paire[1])
    return ', '.join(f'{token} ({compte})' for token, compte in tries[:n])


def comparer_decoupages() -> pd.DataFrame:
    """Tokens vus par chaque découpage, et ceux que seul l'un des deux produit."""
    comptes = {nom: compter_tokens(decouper) for nom, decouper in TOKENISEURS.items()}
    seulement = {
        nom: {token: n for token, n in comptes[nom].items() if token not in comptes[autre]}
        for nom, autre in (('TP1', 'commun'), ('commun', 'TP1'))
    }
    return pd.DataFrame({
        'Tokens distincts': {nom: len(compte) for nom, compte in comptes.items()},
        'Tokens': {nom: sum(compte.values()) for nom, compte in comptes.items()},
        'Tokens distincts absents de l’autre': {
            nom: len(absents) for nom, absents in seulement.items()},
        'Tokens absents de l’autre': {
            nom: sum(absents.values()) for nom, absents in seulement.items()},
        'Les plus fréquents absents de l’autre': {
            nom: plus_frequents(absents) for nom, absents in seulement.items()},
    }).rename_axis('Tokenizer')


def taille_vocabulaire(vectoriseur: TfidfVectorizer) -> int:
    """Nombre de termes appris sur les 769 répliques de développement."""
    return len(vectoriseur.fit(X_train).vocabulary_)


with pd.option_context('display.max_colwidth', None):
    display(comparer_decoupages())

# Empreinte du TP1 : le même TF–IDF sur les mêmes 769 répliques doit retrouver ses 5 915 termes.
termes_tp1 = taille_vocabulaire(TfidfVectorizer(lowercase=True, **PARAMS_TFIDF))
termes_commun = taille_vocabulaire(
    TfidfVectorizer(tokenizer=tokeniser, token_pattern=None, lowercase=False, **PARAMS_TFIDF)
)
print(f'Termes TF–IDF (1-2 grammes, min_df=2) : TP1 {termes_tp1:,} ; '
      f'tokenizer commun {termes_commun:,}')


,Tokens distincts,Tokens,Tokens distincts absents de l’autre,Tokens absents de l’autre,Les plus fréquents absents de l’autre
Tokenizer,,,,,
TP1,5139,36091,129,371,"ll (119), st (56), er (26), ne (13), ld (8), rt (4)"
commun,5449,37652,439,2559,"i (927), a (576), o (151), i'll (103), that's (20), ne'er (13)"


Termes TF–IDF (1-2 grammes, min_df=2) : TP1 5,915 ; tokenizer commun 6,108


> *Mes observations :*
>
> Le tokenizer commun produit 37 652 tokens, dont 5 449 distincts, alors que le découpage du TP1 produit 36 091 tokens, dont 5 139 distincts. Les 439 tokens distincts propres au tokenizer commun sont d'abord `i` (927 après découpage), `a` (576), `o` (151) et `i'll` (103). Les 129 propres au TP1 sont des fragments comme `ll` (119), `st` (56) et `er` (26). Le TF–IDF du TP1 retrouve ses 5 915 termes, donc ce sont bien les mêmes 769 répliques, et le tokenizer commun en donne 6 108.
>
> *Ma conclusion :* le découpage du TP1 effaçait la première personne et les contractions, soit 2 559 tokens sur 36 091 ou 7 % du texte. Le tokenizer commun les rend visibles aux deux familles de modèles, et la section 3.3 mesurera si LinearSVC en profite, première ligne du journal.


## 2. Représentation

### 2.1 Ma référence TF–IDF

Je garde les paramètres du TP1, rassemblés dans `PARAMS_TFIDF` dans la section 1.3, et je construis deux références. Le TF–IDF exact du TP1, avec son `token_pattern` par défaut, assure la continuité avec le TP1, alors que le TF–IDF sur le tokenizer commun rend la comparaison avec les réseaux équitable. Chaque vectoriseur est neuf par fold, car vocabulaire et IDF doivent s'apprendre sur le train du fold seulement.


In [10]:
# Un vectoriseur TF–IDF neuf pour chaque référence, à ajuster sur le train d'un fold.


def construire_tfidf(reference: str) -> TfidfVectorizer:
    """Vectoriseur neuf : 'TP1' = découpage par défaut, 'commun' = tokenizer de la section 1.3."""
    if reference == 'TP1':
        return TfidfVectorizer(lowercase=True, **PARAMS_TFIDF)
    return TfidfVectorizer(tokenizer=tokeniser, token_pattern=None, lowercase=False, **PARAMS_TFIDF)


Pour chaque fold, je mesure la taille du vocabulaire, la densité de la matrice et la part des tokens de validation absents du vocabulaire du train. Cette dernière mesure compte, car elle deviendra le taux de `<UNK>` des embeddings dans la section 2.2. Je garde aussi les unigrammes de chaque fold, pour vérifier dans la section 2.2 qu'ils égalent le vocabulaire des embeddings.


In [11]:
# Bilan de chaque référence sur chaque fold : termes, densité et tokens de validation inconnus.


def pourcentage(partie: float, total: float) -> float:
    """Part en pourcentage."""
    return 100 * partie / total


def detailler_tokens(
    vectoriseur: TfidfVectorizer, decouper, textes_train: pd.Series, textes_validation: pd.Series,
) -> dict:
    """Unigrammes appris, tokens de validation, inconnus et fréquence documentaire du train."""
    unigrammes = {terme for terme in vectoriseur.vocabulary_ if ' ' not in terme}
    tokens_validation = [token for texte in textes_validation for token in decouper(texte)]
    inconnus = [token for token in tokens_validation if token not in unigrammes]  # Absents du train
    documents_train = Counter(token for texte in textes_train for token in set(decouper(texte)))
    return {'unigrammes': unigrammes, 'tokens_validation': tokens_validation,
            'inconnus': inconnus, 'documents_train': documents_train}


def decrire_reference(
    numero: int, reference: str, textes_train: pd.Series, textes_validation: pd.Series,
) -> tuple[dict, dict]:
    """Ajuste une référence sur le train d'un fold : ligne de bilan et détail des tokens."""
    vectoriseur = construire_tfidf(reference)
    matrice_train = vectoriseur.fit_transform(textes_train)  # Appris sur le train seul.
    matrice_validation = vectoriseur.transform(textes_validation)
    assert matrice_train.shape == (len(textes_train), len(vectoriseur.vocabulary_))
    assert matrice_validation.shape == (len(textes_validation), len(vectoriseur.vocabulary_))

    detail = detailler_tokens(vectoriseur, TOKENISEURS[reference], textes_train, textes_validation)
    taille_matrice = matrice_train.shape[0] * matrice_train.shape[1]
    bilan = {
        'Fold': numero, 'Référence': reference,
        'Termes': len(vectoriseur.vocabulary_), 'Unigrammes': len(detail['unigrammes']),
        'Bigrammes': len(vectoriseur.vocabulary_) - len(detail['unigrammes']),
        'Non nuls (%)': pourcentage(matrice_train.nnz, taille_matrice),
        'Tokens de validation inconnus (%)': pourcentage(
            len(detail['inconnus']), len(detail['tokens_validation'])),
    }
    return bilan, detail


# Boucle sur les folds : bilan par référence, unigrammes et détails conservés
bilan_tfidf, details_par_fold = [], {}
unigrammes_tfidf = {}  # (fold, référence) -> unigrammes du train, pour la section 2.2
for numero, fold in enumerate(folds, start=1):
    textes_train, textes_validation = X_train.loc[fold['train']], X_train.loc[fold['validation']]
    details_par_fold[numero] = {}
    for reference in TOKENISEURS:
        bilan, detail = decrire_reference(numero, reference, textes_train, textes_validation)
        bilan_tfidf.append(bilan)
        details_par_fold[numero][reference] = detail
        unigrammes_tfidf[(numero, reference)] = detail['unigrammes']

# Affichages : bilan par fold, puis moyenne par référence
bilan_tfidf = pd.DataFrame(bilan_tfidf).set_index(['Fold', 'Référence'])
display(bilan_tfidf.round(2))
display(bilan_tfidf.groupby('Référence').mean().round(2).rename_axis('Moyenne des cinq folds'))


Termes  Unigrammes  Bigrammes  Non nuls (%)  \
Fold Référence                                                
1    TP1          4861        1996       2865          1.00   
     commun       5025        2038       2987          1.02   
2    TP1          4716        1933       2783          1.01   
     commun       4898        1981       2917          1.02   
3    TP1          4688        1922       2766          1.00   
     commun       4843        1968       2875          1.02   
4    TP1          4644        1945       2699          1.00   
     commun       4806        1993       2813          1.01   
5    TP1          4807        1978       2829          1.01   
     commun       4970        2023       2947          1.02   

                Tokens de validation inconnus (%)  
Fold Référence                                     
1    TP1                                    13.67  
     commun                                 14.12  
2    TP1                                    15.34  
     commun                                 15.54  
3    TP1                                    14.64  
     commun                                 14.98  
4    TP1                                    13.88  
     commun                                 14.28  
5    TP1                                    13.70  
     commun                                 14.20

,Termes,Unigrammes,Bigrammes,Non nuls (%),Tokens de validation inconnus (%)
Moyenne des cinq folds,,,,,
TP1,4743.2,1954.8,2788.4,1.00,14.24
commun,4908.4,2000.6,2907.8,1.02,14.62


> *Mes observations :*
>
> Sur le train d'un fold, soit 615 répliques, le TF–IDF du TP1 compte de 4 644 à 4 861 termes, contre 5 915 sur les 769 répliques de développement. Moins de répliques laissent moins de termes atteindre `min_df=2`. Le tokenizer commun ajoute de 155 à 182 termes par fold, soit 4 908 contre 4 743 en moyenne. Les bigrammes font près de 60 % des termes dans les deux cas. La matrice reste creuse, avec 1,0 % de valeurs non nulles. En validation, 14,2 % des tokens du TP1 et 14,6 % des tokens du tokenizer commun sont absents du vocabulaire du train, de 13,7 % à 15,5 % selon le fold.


Un token de validation sur sept est donc inconnu du train, et je cherche d'où viennent ces inconnus avec deux pistes. Le tokenizer commun garde les contractions et possessifs entiers, comme « father's », qui sont plus rares que leurs fragments. Et `min_df=2` exclut les tokens vus dans une seule réplique du train, donc une partie des inconnus a pourtant été vue.


In [12]:
# Origine des tokens inconnus : apostrophes, tokens vus une seule fois dans le train, et
# unigrammes propres au tokenizer commun.


def vus_une_fois(detail: dict) -> int:
    """Tokens inconnus présents dans une seule réplique du train, donc exclus par min_df=2."""
    return sum(detail['documents_train'][token] == 1 for token in detail['inconnus'])


def analyser_inconnus(numero: int, textes_train: pd.Series, details: dict) -> dict:
    """Origine des inconnus : apostrophes, et tokens vus une seule fois dans le train."""
    tp1, commun = details['TP1'], details['commun']
    sans_apostrophe = [token for token in commun['tokens_validation'] if "'" not in token]
    inconnus_sans_apostrophe = sum(token not in commun['unigrammes'] for token in sans_apostrophe)
    propres_au_commun = commun['unigrammes'] - tp1['unigrammes']
    comptes_train = Counter(token for texte in textes_train for token in tokeniser(texte))
    return {
        'Fold': numero,
        'Inconnus TP1': len(tp1['inconnus']),
        'Inconnus commun': len(commun['inconnus']),
        'Commun avec apostrophe (%)': pourcentage(
            sum("'" in token for token in commun['inconnus']), len(commun['inconnus'])),
        'Commun sans tokens à apostrophe (%)': pourcentage(
            inconnus_sans_apostrophe, len(sans_apostrophe)),
        'Vus une fois dans le train, TP1 (%)': pourcentage(vus_une_fois(tp1), len(tp1['inconnus'])),
        'Vus une fois dans le train, commun (%)': pourcentage(
            vus_une_fois(commun), len(commun['inconnus'])),
        'Unigrammes propres au commun': plus_frequents(
            {token: comptes_train[token] for token in propres_au_commun}, n=5),
    }


detail_inconnus = [
    analyser_inconnus(numero, X_train.loc[fold['train']], details_par_fold[numero])
    for numero, fold in enumerate(folds, start=1)
]
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(detail_inconnus).set_index('Fold').round(2))


,Inconnus TP1,Inconnus commun,Commun avec apostrophe (%),Commun sans tokens à apostrophe (%),"Vus une fois dans le train, TP1 (%)","Vus une fois dans le train, commun (%)",Unigrammes propres au commun
Fold,,,,,,,
1,898,969,9.80,13.05,35.52,35.29,"i (747), a (472), o (128), i'll (87), that's (18)"
2,1097,1157,8.04,14.65,37.65,37.17,"i (748), a (468), o (127), i'll (87), that's (16)"
3,1096,1170,8.29,14.07,38.78,36.84,"i (741), a (451), o (111), i'll (83), that's (13)"
4,1110,1191,8.56,13.36,37.48,36.78,"i (728), a (447), o (121), i'll (75), that's (18)"
5,943,1021,9.30,13.20,36.37,36.04,"i (744), a (466), o (117), i'll (80), that's (15)"


> *Mes observations :*
>
> Entre 8 % et 10 % des inconnus du tokenizer commun contiennent une apostrophe. Sans les tokens à apostrophe, son taux d'inconnus va de 13,1 % à 14,7 %, donc sous celui du TP1 dans chaque fold. De 35 % à 39 % des tokens inconnus apparaissent une seule fois dans le train, car `min_df=2` les a exclus. Les unigrammes propres au tokenizer commun sont les mêmes dans chaque fold, soit `i`, `a`, `o`, `i'll` et `that's`.
>
> *Ma conclusion :* les deux références ont la même taille et la même densité, mais seul le découpage change, donc la section 3.3 isolera l'effet du tokenizer. Les embeddings de la section 2.2 devront absorber un token de validation sur sept avec `<UNK>`, alors que TF–IDF ignore ces tokens sans coût explicite.


### 2.2 Des tokens aux embeddings

Un réseau ne lit pas des mots mais des nombres, donc chaque token reçoit un identifiant, puis une ligne d'une matrice de vecteurs denses, son embedding. Cette matrice s'apprend avec le classifieur par rétropropagation, alors que TF–IDF fixe un poids par terme à partir des fréquences. Je n'utilise pas de vecteurs pré-entraînés, car l'anglais de Shakespeare s'éloigne des corpus modernes et la comparaison avec TF–IDF doit rester à données égales (pas l'objet du TP).


Je construis le vocabulaire sur le train de chaque fold, jamais sur la validation ni le test, car un vocabulaire vu sur la validation ferait passer de l'information de l'évaluation vers l'apprentissage. Je garde un token s'il apparaît dans au moins deux répliques du train, la règle `min_df=2` de TF–IDF. Un token vu une seule fois ne peut pas être appris, car il ajouterait une ligne de paramètres sans exemple pour la régler. Deux identifiants réservés complètent le vocabulaire. `<UNK>` remplace tout token absent, car le réseau doit produire un vecteur à chaque position. `<PAD>` complète les séquences courtes d'un batch, car un tenseur exige des lignes de même longueur.


In [13]:
# Vocabulaire par fold, encodage des répliques en identifiants et encodage des dix classes.
import math

PAD, UNK = '<PAD>', '<UNK>'  # Identifiants 0 et 1, réservés.
CLASSES = sorted(donnees['locuteur'].unique())  # Fixé une fois, ordre alphabétique.
INDICE_CLASSE = {classe: indice for indice, classe in enumerate(CLASSES)}


def construire_vocabulaire(textes_train: pd.Series) -> dict[str, int]:
    """Tokens présents dans au moins deux répliques du train (même règle que min_df=2), triés."""
    documents = Counter(token for texte in textes_train for token in set(tokeniser(texte)))
    retenus = sorted(token for token, nombre in documents.items() if nombre >= 2)
    return {token: indice for indice, token in enumerate([PAD, UNK] + retenus)}


def longueur_maximale(textes_train: pd.Series, quantile: float) -> int:
    """Plafond de longueur : quantile des longueurs en tokens du train, arrondi vers le haut."""
    longueurs = [len(tokeniser(texte)) for texte in textes_train]
    return math.ceil(np.quantile(longueurs, quantile))


def encoder(texte: str, vocabulaire: dict[str, int], longueur_max: int) -> list[int]:
    """Identifiants des tokens, <UNK> pour les absents, tronqués à la longueur maximale."""
    inconnu = vocabulaire[UNK]
    return [vocabulaire.get(token, inconnu) for token in tokeniser(texte)][:longueur_max]


def compter_troncature(textes: pd.Series, longueur_max: int) -> tuple[int, int, int]:
    """Répliques tronquées, tokens perdus et tokens au total pour une longueur maximale."""
    longueurs = [len(tokeniser(texte)) for texte in textes]
    tronquees = sum(longueur > longueur_max for longueur in longueurs)
    perdus = sum(max(longueur - longueur_max, 0) for longueur in longueurs)
    return tronquees, perdus, sum(longueurs)


print('Classes :', ', '.join(f'{indice} = {classe}' for classe, indice in INDICE_CLASSE.items()))


Classes : 0 = CORIOLANUS, 1 = DUKE VINCENTIO, 2 = GLOUCESTER, 3 = ISABELLA, 4 = JULIET, 5 = KING RICHARD II, 6 = LEONTES, 7 = MENENIUS, 8 = PETRUCHIO, 9 = ROMEO


Je vérifie d'abord que ce vocabulaire égale les unigrammes du TF–IDF commun dans chaque fold, car les deux suivent la même règle. Puis je vérifie que le taux de tokens de validation inconnus retrouve celui de la section 2.1.


In [14]:
# Vocabulaire de chaque fold, égal aux unigrammes TF–IDF, et tokens de validation inconnus.


def mesurer_fold(numero: int, fold: dict) -> dict:
    """Vocabulaire du train du fold, égal aux unigrammes TF–IDF, et inconnus de la validation."""
    textes_train, textes_validation = X_train.loc[fold['train']], X_train.loc[fold['validation']]
    vocabulaire = construire_vocabulaire(textes_train)
    assert set(vocabulaire) - {PAD, UNK} == unigrammes_tfidf[(numero, 'commun')]  # Même règle
    tokens = [token for texte in textes_validation for token in tokeniser(texte)]
    inconnus = pourcentage(sum(token not in vocabulaire for token in tokens), len(tokens))
    reference = bilan_tfidf.loc[(numero, 'commun'), 'Tokens de validation inconnus (%)']
    assert abs(inconnus - reference) < 1e-9  # Même mesure que dans la section 2.1
    return {'Fold': numero, 'Vocabulaire (avec <PAD> et <UNK>)': len(vocabulaire),
            'Tokens de validation inconnus (%)': inconnus}


mesures = pd.DataFrame(mesurer_fold(numero, fold) for numero, fold in enumerate(folds, 1))
display(mesures.set_index('Fold').round(2))
print('Vocabulaire égal aux unigrammes TF–IDF du tokenizer commun dans les cinq folds.')


,Vocabulaire (avec <PAD> et <UNK>),Tokens de validation inconnus (%)
Fold,,
1,2040,14.12
2,1983,15.54
3,1970,14.98
4,1995,14.28
5,2025,14.20


Vocabulaire égal aux unigrammes TF–IDF du tokenizer commun dans les cinq folds.


> *Mes observations :*
>
> Le vocabulaire d'un fold compte de 1 970 à 2 040 identifiants, `<PAD>` et `<UNK>` compris, et il égale les unigrammes du TF–IDF commun dans les cinq folds. Le taux de tokens de validation inconnus est le même que dans la section 2.1, de 14,1 % à 15,5 %.


Je fixe ensuite une longueur maximale de séquence, car les répliques vont de 10 à 578 mots. Un plafond borne le coût d'un réseau récurrent, qui lit les tokens un à un, et le padding des batches, mais il coupe des tokens. Je mesure ce coût pour plusieurs quantiles des longueurs du train, en répliques tronquées et en tokens perdus.


In [15]:
# Coût de chaque quantile de longueur : répliques tronquées et tokens perdus, train et validation.
QUANTILES = (0.90, 0.95, 0.99, 1.00)


def mesurer_troncature(fold: dict, quantile: float) -> dict:
    """Longueur maximale sur le train du fold, et ce qu'elle tronque en train et en validation."""
    textes_train, textes_validation = X_train.loc[fold['train']], X_train.loc[fold['validation']]
    longueur_max = longueur_maximale(textes_train, quantile)
    ligne = {'Quantile': quantile, 'Longueur maximale': longueur_max}
    for nom, textes in (('train', textes_train), ('validation', textes_validation)):
        tronquees, perdus, total = compter_troncature(textes, longueur_max)
        ligne[f'Répliques tronquées {nom} (%)'] = pourcentage(tronquees, len(textes))
        ligne[f'Tokens perdus {nom} (%)'] = pourcentage(perdus, total)
    return ligne


troncature = pd.DataFrame(
    mesurer_troncature(fold, quantile) for fold in folds for quantile in QUANTILES)
display(troncature.groupby('Quantile').mean().round(2))


,Longueur maximale,Répliques tronquées train (%),Tokens perdus train (%),Répliques tronquées validation (%),Tokens perdus validation (%)
Quantile,,,,,
0.90,100.6,9.88,15.71,10.66,15.91
0.95,147.0,4.97,8.78,5.07,8.99
0.99,281.0,1.01,2.05,1.17,2.17
1.00,573.6,0.00,0.00,0.13,0.09


> *Mes observations :*
>
> Le quantile 0,90 fixe la longueur maximale à 101 tokens en moyenne, mais tronque 9,9 % des répliques de train et 15,7 % de leurs tokens. Le quantile 0,95 monte à 147 tokens et perd encore 8,8 % des tokens. Le quantile 0,99 monte à 281 tokens, avec 1,0 % des répliques tronquées et 2,1 % des tokens perdus en train, puis 2,2 % en validation. Garder tout, soit 574 tokens, ne tronque plus rien en train.
>
> *Mon choix :* le quantile 0,99 du train du fold, soit 281 tokens en moyenne. Le quantile 0,95 perdrait 8,8 % des tokens, concentrés dans les répliques longues, alors qu'à 0,99 la perte tombe à 2,1 %. Garder tout doublerait la longueur des batches concernés pour 0,1 % des tokens de validation.


Je choisis enfin la dimension des embeddings sur le nombre de paramètres qu'elle crée, car chaque dimension ajoute une colonne à la matrice, donc autant de paramètres que de tokens du vocabulaire. Face aux 615 répliques d'un fold, ce rapport paramètres/exemples chiffre H3, l'hypothèse qu'un réseau mémorise le train sans mieux généraliser.


In [16]:
# Paramètres de la matrice d'embeddings selon sa dimension, rapportés aux répliques de train.
DIMENSIONS = (32, 64, 128)


def parametres_embeddings(taille_vocabulaire: float, dimension: int, repliques: int) -> dict:
    """Paramètres de la matrice d'embeddings et rapport aux répliques de train d'un fold."""
    parametres = taille_vocabulaire * dimension
    return {'Dimension': dimension, 'Paramètres de la matrice': round(parametres),
            'Paramètres par réplique de train': round(parametres / repliques, 1)}


taille_moyenne = mesures['Vocabulaire (avec <PAD> et <UNK>)'].mean()
repliques_train = round(np.mean([len(fold['train']) for fold in folds]))
print(f'Vocabulaire moyen : {taille_moyenne:.0f} identifiants '
      f'pour {repliques_train} répliques de train.')
display(pd.DataFrame(
    parametres_embeddings(taille_moyenne, dimension, repliques_train) for dimension in DIMENSIONS
).set_index('Dimension'))


Vocabulaire moyen : 2003 identifiants pour 615 répliques de train.


,Paramètres de la matrice,Paramètres par réplique de train
Dimension,,
32,64083,104.2
64,128166,208.4
128,256333,416.8


> *Mes observations :*
>
> Avec 2 003 identifiants en moyenne, la matrice d'embeddings compte 64 083 paramètres en dimension 32, 128 166 en dimension 64 et 256 333 en dimension 128. Cela fait 104, 208 et 417 paramètres par réplique de train.
>
> *Mon choix :* la dimension 64, car le rapport atteint déjà 208 paramètres par réplique pour la seule matrice d'embeddings, avant les couches du réseau, et c'est le terrain de H3. La dimension 32 divise ce rapport par deux mais laisse 2 003 tokens sur peu d'axes pour séparer dix locuteurs, alors que la dimension 128 le double sans données supplémentaires.


Je prépare maintenant les batches, car PyTorch consomme des tenseurs et non des listes de tokens. Un `Dataset` encode chaque réplique en identifiants avec la longueur retenue. Puis une fonction d'assemblage complète les séquences d'un batch avec `<PAD>` jusqu'à la plus longue, et garde les longueurs réelles et un masque pour les étapes suivantes.


In [17]:
# Dataset PyTorch et assemblage des batches, avec la longueur et la dimension retenues ci-dessus.
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset

QUANTILE_LONGUEUR = 0.99  # Longueur maximale : quantile 0,99 des longueurs du train du fold.
DIM_EMBEDDING = 64


class JeuRepliques(Dataset):
    """Répliques encodées en identifiants et classes encodées en indices, pour un DataLoader."""

    def __init__(self, textes: pd.Series, cibles: pd.Series, vocabulaire: dict, longueur_max: int):
        self.identifiants = [encoder(texte, vocabulaire, longueur_max) for texte in textes]
        self.classes = [INDICE_CLASSE[cible] for cible in cibles]

    def __len__(self) -> int:
        return len(self.identifiants)

    def __getitem__(self, position: int) -> tuple[torch.Tensor, int]:
        return torch.tensor(self.identifiants[position]), self.classes[position]


def assembler_batch(lot: list) -> dict:
    """Complète les séquences d'un batch avec <PAD> (0) ; garde longueurs, masque et classes."""
    sequences, classes = zip(*lot)
    identifiants = pad_sequence(sequences, batch_first=True, padding_value=0)
    return {'identifiants': identifiants, 'longueurs': torch.tensor([len(s) for s in sequences]),
            'masque': identifiants != 0, 'classes': torch.tensor(classes)}


# Fold 1 : jeu de train et un batch des quatre premières répliques
fold_1 = folds[0]
textes_1, cibles_1 = X_train.loc[fold_1['train']], y_train.loc[fold_1['train']]
vocabulaire_1 = construire_vocabulaire(textes_1)
longueur_max_1 = longueur_maximale(textes_1, QUANTILE_LONGUEUR)
jeu_1 = JeuRepliques(textes_1, cibles_1, vocabulaire_1, longueur_max_1)
batch = assembler_batch([jeu_1[position] for position in range(4)])
print(f'Fold 1 : {len(vocabulaire_1)} identifiants, longueur maximale {longueur_max_1}, '
      f'{len(jeu_1)} répliques de train')
for nom, tenseur in batch.items():
    print(f'   {nom} : forme {tuple(tenseur.shape)}, dtype {tenseur.dtype}')
print(f"   longueurs : {batch['longueurs'].tolist()} ; classes : {batch['classes'].tolist()}\n")


Fold 1 : 2040 identifiants, longueur maximale 280, 615 répliques de train
   identifiants : forme (4, 47), dtype torch.int64
   longueurs : forme (4,), dtype torch.int64
   masque : forme (4, 47), dtype torch.bool
   classes : forme (4,), dtype torch.int64
   longueurs : [31, 47, 10, 40] ; classes : [0, 2, 8, 6]



Il reste à représenter une réplique entière à partir des vecteurs de ses tokens, avec une règle par famille de réseau. Pour le MLP, je prends la moyenne des embeddings, qui ignore l'ordre, car « Romeo loves Juliet » et « Juliet loves Romeo » ont la même moyenne. J'exclus le padding de cette moyenne, sinon les répliques courtes seraient diluées par des vecteurs vides. Pour le LSTM et le GRU, je garde l'état récurrent après le dernier token réel, car lire des `<PAD>` modifierait l'état sans apporter d'information. Je vérifie ces deux règles avec un test, car ajouter du padding ne doit rien changer.


In [18]:
# Deux représentations d'une réplique entière et le test d'invariance au padding.
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

PADDING_SUPPLEMENTAIRE = 20  # Tokens <PAD> ajoutés pour le test d'invariance.


def moyenne_masquee(vecteurs: torch.Tensor, masque: torch.Tensor) -> torch.Tensor:
    """Moyenne des vecteurs des tokens réels seulement : le padding ne compte pas."""
    poids = masque.unsqueeze(-1).to(vecteurs.dtype)
    return (vecteurs * poids).sum(dim=1) / poids.sum(dim=1)


def etat_final(rnn: nn.Module, vecteurs: torch.Tensor, longueurs: torch.Tensor) -> torch.Tensor:
    """État récurrent après le dernier token réel ; les directions sont concaténées."""
    paquet = pack_padded_sequence(vecteurs, longueurs.cpu(), batch_first=True, enforce_sorted=False)
    _, etats = rnn(paquet)
    if isinstance(etats, tuple):  # LSTM : (h, c) ; je garde h.
        etats = etats[0]
    directions = 2 if rnn.bidirectional else 1
    return torch.cat(list(etats[-directions:]), dim=-1)  # Dernière couche, chaque direction


def representer(identifiants: torch.Tensor, embeddings: nn.Embedding, rnns: dict) -> dict:
    """Moyenne masquée et états finaux d'un batch, pour des modules donnés."""
    identifiants = identifiants.to(device)
    masque = identifiants != 0
    vecteurs = embeddings(identifiants)
    sorties = {'Moyenne masquée': moyenne_masquee(vecteurs, masque)}
    for nom, rnn in rnns.items():
        sorties[nom] = etat_final(rnn, vecteurs, masque.sum(dim=1))
    return sorties


def verifier_invariance(texte: str, vocabulaire: dict, longueur_max: int) -> pd.DataFrame:
    """La même réplique avec des <PAD> en plus doit donner les mêmes représentations."""
    torch.manual_seed(RANDOM_STATE)  # Modules aléatoires mais reproductibles
    embeddings = nn.Embedding(len(vocabulaire), DIM_EMBEDDING, padding_idx=0).to(device)
    rnns = {
        'GRU unidirectionnel': nn.GRU(DIM_EMBEDDING, 8, batch_first=True).to(device),
        'GRU bidirectionnel': nn.GRU(
            DIM_EMBEDDING, 8, batch_first=True, bidirectional=True).to(device),
    }
    identifiants = torch.tensor([encoder(texte, vocabulaire, longueur_max)])
    supplement = torch.zeros(1, PADDING_SUPPLEMENTAIRE, dtype=torch.long)
    sans = representer(identifiants, embeddings, rnns)
    avec = representer(torch.cat([identifiants, supplement], dim=1), embeddings, rnns)
    return pd.DataFrame([
        {'Représentation': nom, 'Dimension': sans[nom].shape[1],
         'Écart maximal': (sans[nom] - avec[nom]).abs().max().item(),
         'allclose': torch.allclose(sans[nom], avec[nom], atol=1e-6)}
        for nom in sans
    ]).set_index('Représentation')


# Invariance au padding sur la première réplique de train du fold 1
display(verifier_invariance(textes_1.iloc[0], vocabulaire_1, longueur_max_1))


,Dimension,Écart maximal,allclose
Représentation,,,
Moyenne masquée,64,2.980232e-08,True
GRU unidirectionnel,8,0.000000e+00,True
GRU bidirectionnel,16,0.000000e+00,True


> *Ma conclusion :* sur le fold 1, la longueur maximale vaut 280 tokens et un batch de quatre répliques se complète à 47 positions, la longueur de sa plus longue réplique. La moyenne masquée et l'état final d'un GRU, unidirectionnel ou bidirectionnel, ne changent pas quand j'ajoute 20 `<PAD>`. L'écart maximal vaut 3 × 10⁻⁸ pour la moyenne et 0 pour les états. La section 3 reçoit donc `JeuRepliques`, `assembler_batch`, `moyenne_masquee` et `etat_final`, avec `QUANTILE_LONGUEUR = 0.99` et `DIM_EMBEDDING = 64`.


## 3. Comparaison entre modèles ML et DL

### 3.1 Préparer l'entraînement commun

Je compare la classe majoritaire, LinearSVC, puis MLP, LSTM et GRU. J'entraîne les réseaux avec la cross-entropy, car la cible est une classe parmi dix, et je les compare avec le F1 macro, car chaque classe compte autant.


Une epoch est un passage complet sur le train du fold. Le nombre d'epochs devient un hyperparamètre choisi par validation croisée, comme `C` au TP1. Chaque réseau s'entraîne un nombre maximal d'epochs dans chaque fold, puis je retiens l'epoch où la courbe moyenne de validation des cinq folds est la plus haute, l'epoch retenue. Une validation interne de 123 répliques, soit 10 à 16 par classe, aurait rendu ce choix bruité et retiré 20 % des données aux réseaux, alors que LinearSVC aurait gardé les 615. L'epoch retenue est la plus petite epoch à moins de 0,5 point du maximum de cette courbe moyenne, car un plateau atteint tôt vaut mieux qu'un pic tardif et bruité.


In [19]:
# Règle de l'epoch retenue : la plus petite epoch à moins de 0,5 point du maximum de la courbe.
TOLERANCE_EPOCH = 0.005  # 0,5 point de F1 macro, sur une échelle de 0 à 1


def epoch_retenue(courbe: pd.Series, tolerance: float = TOLERANCE_EPOCH) -> int:
    """Première epoch dont le F1 macro moyen de validation atteint le maximum moins la tolérance."""
    seuil = courbe.max() - tolerance
    return int(courbe.index[courbe >= seuil][0])


# Exemple construit : le maximum est à l'epoch 5, mais l'epoch 4 est déjà à 0,5 point de lui.
exemple = pd.Series([0.30, 0.42, 0.47, 0.476, 0.48, 0.47],
                    index=pd.RangeIndex(1, 7, name='Epoch'))
print(f'Courbe : {exemple.round(3).to_dict()}')
print(f'Epoch retenue : {epoch_retenue(exemple)}, '
      f'pour un maximum de {exemple.max():.3f} à l’epoch {exemple.idxmax()}')


Courbe : {1: 0.3, 2: 0.42, 3: 0.47, 4: 0.476, 5: 0.48, 6: 0.47}
Epoch retenue : 4, pour un maximum de 0.480 à l’epoch 5


Je fixe d'abord les réglages communs aux trois réseaux, car une comparaison n'a de sens que si seul le modèle change. Ils tiennent dans un dictionnaire `REGLAGES`, donc une expérience de la section 4.3 en change une seule clé.

| Réglage | Valeur de départ | Pourquoi |
|---|---|---|
| Loss | cross-entropy | Dix classes exclusives. |
| Optimiseur | Adam, taux d'apprentissage 1e-3 | Pas adaptatif par paramètre, robuste sans réglage fin. |
| Batch | 32 répliques | Une vingtaine de mises à jour par epoch sur 615 répliques. |
| Epochs | 40 au maximum, puis epoch retenue sur la courbe moyenne de validation | L'epoch est un hyperparamètre choisi par validation croisée, comme `C` au TP1. |
| Dropout, weight decay | 0, puis testés dans la section 4.3 | Je mesure d'abord le sur-apprentissage sans le corriger. |
| Gradient clipping | norme du gradient plafonnée à 1 | Évite les gradients qui explosent dans les séquences longues. |
| Pondération des classes | aucune | Déséquilibre modéré, de 78 à 129, et le F1 macro surveille chaque classe. |
| Seeds | 42, réinitialisée à chaque fold, et 3 seeds si le temps le permet | Séparer la dispersion des folds de celle de l'initialisation. |


In [20]:
# Réglages communs à tous les réseaux, seed et chargeurs de données d'un fold.
from torch.utils.data import DataLoader

REGLAGES = {
    'taille_batch': 32,
    'taux_apprentissage': 1e-3,
    'epochs_max': 40,
    'dropout': 0.0,
    'weight_decay': 0.0,
    'clip_gradient': 1.0,  # Norme maximale du gradient
    'largeur_cachee': 64,  # Couche cachée du MLP et état des réseaux récurrents
    'dim_embedding': DIM_EMBEDDING,
    'quantile_longueur': QUANTILE_LONGUEUR,
    'tolerance_epoch': TOLERANCE_EPOCH,
    'seeds': (RANDOM_STATE,),
    'device': device,
}


def fixer_seed(seed: int) -> None:
    """Réinitialise random, numpy et torch, y compris MPS, avant chaque entraînement."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


def construire_chargeur(jeu: JeuRepliques, reglages: dict, melanger: bool, seed: int) -> DataLoader:
    """DataLoader avec assemblage des batches, mélange du train fixé par la seed."""
    generateur = torch.Generator().manual_seed(seed) if melanger else None
    return DataLoader(jeu, batch_size=reglages['taille_batch'], shuffle=melanger,
                      generator=generateur, collate_fn=assembler_batch)


def preparer_fold(fold: dict, reglages: dict, seed: int) -> dict:
    """Vocabulaire et longueur maximale sur le train du fold, puis les deux chargeurs."""
    textes_train, textes_validation = X_train.loc[fold['train']], X_train.loc[fold['validation']]
    vocabulaire = construire_vocabulaire(textes_train)
    longueur_max = longueur_maximale(textes_train, reglages['quantile_longueur'])
    jeu_train = JeuRepliques(textes_train, y_train.loc[fold['train']], vocabulaire, longueur_max)
    jeu_validation = JeuRepliques(
        textes_validation, y_train.loc[fold['validation']], vocabulaire, longueur_max)
    return {'vocabulaire': vocabulaire, 'longueur_max': longueur_max,
            'train': construire_chargeur(jeu_train, reglages, True, seed),
            'validation': construire_chargeur(jeu_validation, reglages, False, seed)}


print('\n'.join(f'{cle} : {valeur}' for cle, valeur in REGLAGES.items()))


taille_batch : 32
taux_apprentissage : 0.001
epochs_max : 40
dropout : 0.0
weight_decay : 0.0
clip_gradient : 1.0
largeur_cachee : 64
dim_embedding : 64
quantile_longueur : 0.99
tolerance_epoch : 0.005
seeds : (42,)
device : mps


Les trois réseaux partagent un modèle de base et une boucle d'entraînement, car seule la représentation d'une réplique doit changer entre eux. Le modèle de base tient les embeddings, le dropout et la couche de sortie à dix scores, et chaque réseau ne définit que sa représentation. La boucle enchaîne forward, cross-entropy, backward, gradient plafonné et mise à jour, puis mesure loss et F1 macro sur train et validation à chaque epoch, en mode évaluation pour un score exact. Elle garde les prédictions de validation de chaque epoch, car l'epoch est choisie après coup sur la courbe moyenne.


In [21]:
# Un modèle de base pour les trois réseaux et une boucle d'entraînement commune.
import math
import time

import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score


class ClassifieurTokens(nn.Module):
    """Embeddings partagés, une représentation définie par la sous-classe, dix scores en sortie."""

    def __init__(self, taille_vocabulaire: int, reglages: dict, dim_representation: int):
        super().__init__()
        self.embeddings = nn.Embedding(taille_vocabulaire, reglages['dim_embedding'], padding_idx=0)
        self.dropout = nn.Dropout(reglages['dropout'])
        self.sortie = nn.Linear(dim_representation, len(CLASSES))

    def representer(self, vecteurs, masque, longueurs) -> torch.Tensor:
        raise NotImplementedError

    def forward(self, batch: dict) -> torch.Tensor:
        vecteurs = self.dropout(self.embeddings(batch['identifiants']))
        representation = self.representer(vecteurs, batch['masque'], batch['longueurs'])
        return self.sortie(self.dropout(representation))


class MoyenneLineaire(ClassifieurTokens):
    """Modèle minimal pour les contrôles : moyenne masquée puis couche de sortie."""

    def __init__(self, taille_vocabulaire: int, reglages: dict):
        super().__init__(taille_vocabulaire, reglages, reglages['dim_embedding'])

    def representer(self, vecteurs, masque, longueurs):
        return moyenne_masquee(vecteurs, masque)


def vers_device(batch: dict, device: torch.device) -> dict:
    """Envoie un batch sur le device, sauf les longueurs, que le packing attend sur CPU."""
    return {cle: valeur if cle == 'longueurs' else valeur.to(device)
            for cle, valeur in batch.items()}


def entrainer_epoch(modele, chargeur, optimiseur, reglages: dict) -> float:
    """Une epoch : forward, cross-entropy, backward, gradient plafonné, mise à jour des poids."""
    modele.train()
    total = 0.0
    for batch in chargeur:
        batch = vers_device(batch, reglages['device'])
        optimiseur.zero_grad()
        loss = F.cross_entropy(modele(batch), batch['classes'])
        loss.backward()
        nn.utils.clip_grad_norm_(modele.parameters(), reglages['clip_gradient'])
        optimiseur.step()
        total += loss.item() * len(batch['classes'])
    return total / len(chargeur.dataset)


@torch.no_grad()
def evaluer_modele(modele, chargeur, device: torch.device) -> dict:
    """Loss, prédictions en indices de classe, F1 macro et accuracy, en mode évaluation."""
    modele.eval()
    total, predictions, vrais = 0.0, [], []
    for batch in chargeur:
        batch = vers_device(batch, device)
        scores_batch = modele(batch)
        total += F.cross_entropy(scores_batch, batch['classes']).item() * len(batch['classes'])
        predictions += scores_batch.argmax(dim=1).tolist()
        vrais += batch['classes'].tolist()
    return {'loss': total / len(chargeur.dataset), 'predictions': predictions,
            'f1': f1_score(vrais, predictions, average='macro', zero_division=0),
            'accuracy': accuracy_score(vrais, predictions)}


def entrainer_reseau(construire_modele, fold: dict, reglages: dict, seed: int) -> dict:
    """Entraîne un réseau sur un fold pendant epochs_max, avec historique et prédictions."""
    fixer_seed(seed)
    donnees = preparer_fold(fold, reglages, seed)
    modele = construire_modele(len(donnees['vocabulaire']), reglages).to(reglages['device'])
    optimiseur = torch.optim.Adam(modele.parameters(), lr=reglages['taux_apprentissage'],
                                  weight_decay=reglages['weight_decay'])
    historique, predictions, debut = [], {}, time.perf_counter()
    for epoch in range(1, reglages['epochs_max'] + 1):
        entrainer_epoch(modele, donnees['train'], optimiseur, reglages)
        train = evaluer_modele(modele, donnees['train'], reglages['device'])
        validation = evaluer_modele(modele, donnees['validation'], reglages['device'])
        historique.append({
            'epoch': epoch, 'loss train': train['loss'], 'F1 train': train['f1'],
            'loss validation': validation['loss'], 'F1 validation': validation['f1']})
        predictions[epoch] = validation['predictions']
    return {'historique': pd.DataFrame(historique).set_index('epoch'), 'predictions': predictions,
            'parametres': sum(p.numel() for p in modele.parameters()),
            'temps': time.perf_counter() - debut, 'modele': modele}


Avant les cinq folds, je contrôle la boucle sur le fold 1 avec un modèle minimal, moyenne masquée puis couche de sortie, car une erreur coûterait cinq folds fois trois réseaux. La loss initiale doit approcher ln 10 ≈ 2,303, car dix classes presque équiprobables donnent cette valeur, et les gradients doivent être finis. Un seul batch répété doit être sur-appris jusqu'à une loss proche de 0, sinon la boucle n'apprend pas. Enfin deux runs à seed égale doivent donner les mêmes prédictions, sinon la seed ne contrôle pas le hasard.


In [22]:
# Contrôles sur le fold 1 : loss initiale, gradients, un batch sur-appris, puis déterminisme.
PAS_SUR_UN_BATCH = 1000
EPOCHS_CONTROLE = 3


def verifier_batch(fold: dict, reglages: dict) -> pd.DataFrame:
    """Loss initiale face à ln 10, norme du gradient, puis loss après des pas sur un seul batch."""
    fixer_seed(RANDOM_STATE)
    donnees = preparer_fold(fold, reglages, RANDOM_STATE)
    modele = MoyenneLineaire(len(donnees['vocabulaire']), reglages).to(reglages['device'])
    batch = vers_device(next(iter(donnees['train'])), reglages['device'])
    optimiseur = torch.optim.Adam(modele.parameters(), lr=reglages['taux_apprentissage'])
    loss = F.cross_entropy(modele(batch), batch['classes'])
    loss.backward()
    norme = nn.utils.clip_grad_norm_(modele.parameters(), float('inf')).item()  # Sans plafond
    lignes = [
        {'Contrôle': 'Loss initiale sur un batch', 'Valeur': loss.item(), 'Attendu': math.log(10)},
        {'Contrôle': 'Norme du gradient', 'Valeur': norme, 'Attendu': 'finie'}]
    for _ in range(PAS_SUR_UN_BATCH):
        optimiseur.zero_grad()
        loss = F.cross_entropy(modele(batch), batch['classes'])
        loss.backward()
        optimiseur.step()
    lignes.append({'Contrôle': f'Loss après {PAS_SUR_UN_BATCH} pas sur ce batch',
                   'Valeur': loss.item(), 'Attendu': 'proche de 0'})
    return pd.DataFrame(lignes).set_index('Contrôle')


def verifier_determinisme(fold: dict, reglages: dict) -> pd.DataFrame:
    """Deux runs à seed égale : mêmes prédictions et même F1 après quelques epochs."""
    essai = dict(reglages, epochs_max=EPOCHS_CONTROLE)
    runs = [entrainer_reseau(MoyenneLineaire, fold, essai, RANDOM_STATE) for _ in range(2)]
    return pd.DataFrame([{
        'Contrôle': f'Deux runs à seed égale, {EPOCHS_CONTROLE} epochs',
        'F1 validation run 1': runs[0]['historique'].loc[EPOCHS_CONTROLE, 'F1 validation'],
        'F1 validation run 2': runs[1]['historique'].loc[EPOCHS_CONTROLE, 'F1 validation'],
        'Prédictions identiques': runs[0]['predictions'] == runs[1]['predictions'],
    }]).set_index('Contrôle')


display(verifier_batch(folds[0], REGLAGES).round(4))
display(verifier_determinisme(folds[0], REGLAGES).round(4))


,Valeur,Attendu
Contrôle,,
Loss initiale sur un batch,2.3042,2.302585
Norme du gradient,0.3741,finie
Loss après 1000 pas sur ce batch,0.0072,proche de 0


,F1 validation run 1,F1 validation run 2,Prédictions identiques
Contrôle,,,
"Deux runs à seed égale, 3 epochs",0.1508,0.1508,True


> *Mes observations :*
>
> La loss initiale vaut 2,304 pour un attendu de 2,303, donc le modèle part bien d'une hésitation uniforme entre les dix classes, et la norme du gradient, 0,374, est finie. Après 1 000 pas sur un seul batch, la loss tombe à 0,007, donc la boucle apprend. Deux runs à seed égale donnent les mêmes prédictions et le même F1 macro de 0,151 après 3 epochs, donc la seed contrôle bien le hasard.


Je termine par deux fonctions d'évaluation, une pour les pipelines scikit-learn et une pour les réseaux, car ML et DL doivent passer par la même boucle de folds et le même scoreur. Chacune range dans un registre `resultats` les scores par fold, le temps et les prédictions de validation par indice de réplique, car la section 4 comparera les modèles réplique par réplique. Pour un réseau, la fonction entraîne chaque fold, puis choisit l'epoch retenue sur la courbe moyenne et score cette epoch.


In [23]:
# Évaluer un modèle ML ou un réseau sur les mêmes folds, avec le même scoreur, dans un registre.
resultats = {}  # nom -> scores par fold, prédictions de validation par indice, coût


def scores(vrais, predits) -> dict:
    """F1 macro et accuracy, les deux métriques de comparaison."""
    return {'F1 macro': f1_score(vrais, predits, average='macro', zero_division=0),
            'Accuracy': accuracy_score(vrais, predits)}


def resumer(nom: str) -> pd.DataFrame:
    """Moyenne et écart-type des scores par fold d'un modèle du registre."""
    table = resultats[nom]['folds'][['F1 train', 'F1 macro', 'Accuracy', 'Temps (s)']]
    return pd.DataFrame({'moyenne': table.mean(), 'écart-type': table.std()}).T.round(3)


def evaluer_ml(nom: str, construire_pipeline) -> pd.DataFrame:
    """Ajuste un pipeline neuf sur chaque fold, garde scores, temps et prédictions par indice."""
    lignes, predictions = [], pd.Series(index=X_train.index, dtype=object)
    for numero, fold in enumerate(folds, start=1):
        debut = time.perf_counter()
        textes, cibles = X_train.loc[fold['train']], y_train.loc[fold['train']]
        pipeline = construire_pipeline().fit(textes, cibles)
        predit = pipeline.predict(X_train.loc[fold['validation']])
        predictions.loc[fold['validation']] = predit
        f1_train = scores(cibles, pipeline.predict(textes))['F1 macro']
        lignes.append({'Fold': numero, 'F1 train': f1_train,
                       **scores(y_train.loc[fold['validation']], predit),
                       'Temps (s)': time.perf_counter() - debut})
    resultats[nom] = {'famille': 'ML', 'folds': pd.DataFrame(lignes).set_index('Fold'),
                      'predictions': predictions}
    return resultats[nom]['folds'].round(3)


def choisir_epoch(runs: list, reglages: dict) -> int:
    """Epoch retenue sur la courbe moyenne de validation de tous les runs (folds et seeds)."""
    courbe = pd.concat([run['historique']['F1 validation'] for run in runs], axis=1).mean(axis=1)
    return epoch_retenue(courbe, reglages['tolerance_epoch'])


def evaluer_reseau(nom: str, construire_modele, reglages: dict = REGLAGES) -> pd.DataFrame:
    """Entraîne un réseau sur chaque fold et chaque seed, puis score l'epoch retenue."""
    runs = [dict(entrainer_reseau(construire_modele, fold, reglages, seed),
                 fold=numero, seed=seed)
            for numero, fold in enumerate(folds, start=1)
            for seed in reglages['seeds']]
    epoch = choisir_epoch(runs, reglages)
    lignes, predictions = [], pd.Series(index=X_train.index, dtype=object)
    for run in runs:
        fold = folds[run['fold'] - 1]
        predit = [CLASSES[indice] for indice in run['predictions'][epoch]]
        if run['seed'] == reglages['seeds'][0]:
            predictions.loc[fold['validation']] = predit  # Prédictions de la première seed
        lignes.append({'Fold': run['fold'], 'Seed': run['seed'],
                       'F1 train': run['historique'].loc[epoch, 'F1 train'],
                       **scores(y_train.loc[fold['validation']], predit),
                       'Temps (s)': run['temps']})
    resultats[nom] = {'famille': 'DL', 'folds': pd.DataFrame(lignes).set_index(['Fold', 'Seed']),
                      'predictions': predictions, 'epoch': epoch, 'runs': runs,
                      'parametres': runs[0]['parametres']}
    return resultats[nom]['folds'].round(3)


print('Registre prêt : evaluer_ml, evaluer_reseau et resumer écrivent dans resultats.')


Registre prêt : evaluer_ml, evaluer_reseau et resumer écrivent dans resultats.


> *Ma conclusion :* la section 3 dispose des réglages `REGLAGES`, d'une seed réinitialisée à chaque run et d'un modèle de base `ClassifieurTokens` où chaque réseau ne définit que sa représentation. La boucle est contrôlée sur un batch et sur le déterminisme, et les deux évaluateurs `evaluer_ml` et `evaluer_reseau` écrivent scores, temps et prédictions par réplique dans `resultats`. Chaque modèle des sections 3.2 à 3.6 passe donc par le même chemin.


### 3.2 Baseline majoritaire

> *Mon hypothèse :* prédire la classe la plus fréquente du train donne un repère faible mais nécessaire, car un modèle textuel doit apporter un gain mesuré par rapport à ce repère.


In [24]:
# TODO : évaluer DummyClassifier(strategy="most_frequent") sur les cinq folds.
# TODO : conserver les prédictions, l’accuracy et le F1 macro de chaque fold.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.3 TF–IDF + LinearSVC

> *Mon hypothèse :* sur ce petit jeu, les indices lexicaux restent compétitifs face aux réseaux, car LinearSVC convient aux vecteurs TF–IDF creux. Le tokenizer commun aide un peu, car « I » et les contractions deviennent visibles, mais sans changer l'ordre de grandeur.

Je reprends `C=1`, `class_weight=None`, `max_iter=5000` et `random_state=42`, puis j'évalue les deux références de la section 2.1 sur les folds communs. Leur écart mesure l'effet du tokenizer seul, première ligne du journal. Je regarde le F1 macro et l'accuracy par fold, leur moyenne et leur écart-type, l'écart train/validation et le temps.


In [25]:
# TODO : évaluer les deux Pipelines TF–IDF + LinearSVC sur les folds communs.
# TODO : conserver les prédictions par indice, les scores train/validation et les temps.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.4 Embeddings + moyenne + MLP

> *Mon hypothèse :* un petit MLP apprend des combinaisons utiles d'indices après la moyenne des embeddings, sans l'ordre des mots. Si H1 tient, son F1 macro reste à moins de 0,05 de LinearSVC.

Une couche cachée ReLU puis dix scores de classe, et je justifierai sa largeur par le nombre de paramètres et le sur-apprentissage observé. Je regarde les courbes loss et F1 macro train/validation, l'epoch retenue, l'écart train/validation à cette epoch, le F1 macro par fold, les paramètres et le temps.


In [26]:
# TODO : construire embeddings, moyenne des embeddings hors padding, couche cachée ReLU et sortie à
#   10 classes.
# TODO : entraîner dans chaque fold avec la boucle commune, 40 epochs au maximum, courbes
#   enregistrées.
# TODO : choisir l'epoch retenue sur la courbe moyenne de validation ; conserver prédictions,
#   courbes et temps.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.5 Embeddings + LSTM

> *Mon hypothèse :* lire la séquence apporte des indices que la moyenne perd, malgré le faible nombre de répliques. Si H2 tient, le LSTM dépasse le MLP et le mélange des mots de la section 4.3 le fait redescendre.

Une couche unidirectionnelle, puis une couche de sortie linéaire à dix scores. Je garde une version bidirectionnelle pour la section 4.3, car classer une réplique entière n'impose aucune contrainte causale. Je regarde les mêmes éléments que pour le MLP, plus la part de répliques tronquées et la norme des gradients.


In [27]:
# TODO : construire embeddings, LSTM à une couche et couche de sortie linéaire à 10 classes.
# TODO : utiliser les longueurs réelles pour exclure le padding de l'état retenu.
# TODO : évaluer avec la boucle, les folds, le budget d'epochs et la règle d'epoch retenue communs.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.6 Embeddings + GRU face au LSTM

> *Mon hypothèse :* le GRU suffit pour nos répliques avec moins de paramètres qu'un LSTM de même largeur (H4), donc je compare résultats et coût avant de choisir.

Même couche unidirectionnelle, même dimension d'embedding, même largeur cachée et même couche de sortie que le LSTM. Tokenisation, vocabulaire, initialisation des embeddings et réglages restent communs, donc seuls les poids diffèrent. Une largeur égale ne donne pas un nombre de paramètres égal, donc je les compte, puis je compare F1 macro, dispersion, temps et écart train/validation, fold par fold face au LSTM.


In [28]:
# TODO : remplacer le LSTM par un GRU en conservant les autres choix de cette comparaison.
# TODO : vérifier les paramètres entraînables et le traitement du padding.
# TODO : conserver prédictions, courbes, epoch retenue et temps sur les mêmes folds.


> *Mes observations :*
>
> *Ma conclusion :*


## 4. Évaluation, erreurs et conclusions

### 4.1 Comparer la généralisation

Je rassemble accuracy et F1 macro par fold, puis moyenne et écart-type. La dispersion décrit cinq scores, mais elle ne constitue pas un test de significativité.

Je compare les modèles sur les mêmes répliques. D'abord par la différence de F1 macro fold par fold entre chaque réseau et LinearSVC, avec son signe. Puis par le nombre de répliques où un modèle a raison et l'autre tort, sur les 769 prédictions de validation. Je lis les courbes train/validation et jamais le score train seul, puis les métriques par classe, les paramètres et le temps total. Chaque matrice de confusion est commentée.


In [29]:
# TODO : afficher les scores par fold, les moyennes, les écarts-types, les paramètres et les temps.
# TODO : calculer les différences par fold avec LinearSVC et les comptes « A juste, B faux » sur les
#   769 prédictions.
# TODO : tracer et commenter les courbes train/validation enregistrées ; marquer l'epoch retenue.
# TODO : afficher précision, rappel, F1 et supports par classe, puis les matrices de confusion
#   commentées.


### 4.2 Revenir aux répliques

Je lis les erreurs communes et les désaccords entre ML et DL, en suivant JULIET–ROMEO et GLOUCESTER–DUKE VINCENTIO sans supposer que les erreurs du TP1 se reproduisent.

Le TP1 (section 8.3) suggérait que le modèle reconnaît la pièce plutôt que le style, donc je le teste. Je regroupe les dix locuteurs par pièce à partir du corpus, car les locuteurs qui se répondent dans une fenêtre de répliques consécutives appartiennent à la même pièce. Puis je compare la part d'erreurs intra-pièce à la part inter-pièce et au hasard, car JULIET–ROMEO est intra-pièce alors que GLOUCESTER–DUKE VINCENTIO est inter-pièce.

Quelles erreurs le DL corrige-t-il et lesquelles introduit-il ? Les noms, le vocabulaire partagé, la longueur ou les tokens inconnus donnent-ils une explication vérifiable ?


In [30]:
# TODO : afficher des répliques entières avec le label réel et les prédictions des modèles.
# TODO : reconstruire les pièces par voisinage de locuteurs dans le corpus ; mesurer erreurs intra-
#   pièce et inter-pièce.
# TODO : examiner cinq cas concrets, les confusions ciblées et les supports correspondants.


> *Mes observations :*
>
> *Ma conclusion :*


### 4.3 Expériences ciblées, si les résultats le justifient

Un seul facteur change à la fois, une clé de `REGLAGES` ou une fonction de prétraitement, puis je réentraîne dans les folds avant le test. Par priorité :

1. **Ordre des mots.** Mélanger les mots après tokenisation avec un générateur fixé par l'indice de la réplique, pour que ML et DL voient la même perturbation. Si H2 tient, LSTM et GRU baissent mais MLP et unigrammes TF–IDF ne bougent pas.
2. **Noms propres.** Masquer une liste de noms choisie sur le train du fold seulement, car `kate` et `paulina` motivent la question depuis le TP1. Je lis le résultat par classe, car les classes dont le F1 chute le plus sont reconnues par leurs noms.
3. **Régularisation.** Dropout et weight decay activés, contre le réglage de départ.
4. **Bidirectionnel.** LSTM ou GRU bidirectionnel, à largeur égale.
5. **Quantité de données.** 25, 50, 75 et 100 % du train de chaque fold pour LinearSVC et le meilleur réseau. C'est la seule expérience qui dit si le deep learning manque de données ici.

Une comparaison MLP/LSTM/GRU ne suffit pas à attribuer un gain à l'ordre, et masquer quelques noms ne démontre pas que le signal restant est du style.


In [31]:
# TODO : choisir les expériences à mener sur les folds de développement, avant le test final.
# TODO : figer la perturbation ou la clé de REGLAGES modifiée, garder le reste du protocole et
#   comparer au modèle non modifié.


> *Mes observations :*
>
> *Ma conclusion :*


### 4.4 Figer les choix et évaluer sur le test

Je choisis les configurations sur le F1 macro moyen en validation, avec leur dispersion et leur coût. Chaque réseau retenu prend son epoch retenue, reconstruit son vocabulaire sur les 769 répliques et se réentraîne depuis une nouvelle initialisation, sans utiliser le test pour s'arrêter. Une epoch sur 769 répliques compte 25 % de mises à jour de plus qu'une epoch de fold, et je le signale sans le corriger.

Les références ML sont réajustées sur les 769 répliques. Je compare les choix figés sur les 193 répliques de test, avec le repère du TP1 de 0,563 de F1 macro pour LinearSVC, puis j'analyse les erreurs sans relancer de réglage.

Ce test a déjà servi au TP1 pour choisir LinearSVC, donc la comparaison reste exploratoire, et les hypothèses tirées de ses erreurs ne le rendent pas indépendant.


In [32]:
# TODO : figer les configurations et l'epoch retenue ; reconstruire vocabulaires et modèles sur les
#   769 répliques.
# TODO : évaluer les choix figés sur les 193 répliques réservées et conserver leurs prédictions.
# TODO : comparer scores, matrices de confusion commentées, erreurs et coût dans un tableau final.


> *Mes observations :*
>
> *Ma conclusion :*


### 4.5 Mon journal d'expériences

Une ligne après chaque essai, et une hypothèse reste même si le résultat ne la confirme pas.

| Expérience | Hypothèse | Changement | Observation chiffrée | Décision |
|---|---|---|---|---|
| 1. Tokenizer commun (section 3.3) | « I » et les contractions aident LinearSVC | `token_pattern` par défaut remplacé par `tokeniser` | | |

### 4.6 Ce que je peux conclure

Je réponds à quatre questions, soit quel signal je prédis, ce que chaque modèle apprend réellement, sur quelles données il généralise et quelle métrique traduit le besoin. Quel modèle répond le mieux au problème, scores et coût compris ? Quelles hypothèses, de H1 à H4, mes expériences soutiennent-elles ? Qu'est-ce qui reste incertain sur le contexte, le style et la généralisation ?

> *Ma conclusion :*

### Pour aller plus loin

Prochaines pistes, hors de ce TP, avec un encodeur pré-entraîné de type BERT pour classer les répliques, et l'attention au niveau des caractères sur ce même corpus.
